In [1]:
import matplotlib.pyplot as plt
import numpy as np
import sumo_rl
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from sumo_rl.environment.observations import ObservationFunction
from gymnasium import spaces
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# Monkey patch to fix agent_selector compatibility
# # Replace your current monkey-patching block with this:
# from pettingzoo.utils.agent_selector import agent_selector as agent_selector_class
# import pettingzoo.utils

# # If you MUST monkey patch, point it to the class, not the module
# pettingzoo.utils.agent_selector = agent_selector_class

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [2]:
# Hyperparameters
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.97
batch_size = 64
learning_rate = 1e-3
num_seconds = 1000
episodes = 50
replay_buffer_size = 10000
save_interval = 10  # Save models every 10 episodes

In [3]:
def priority_reward_fn(ts):
    """
    Custom reward function prioritizing emergency vehicles.
    Penalizes stopped ambulances heavily, buses moderately, and normal cars lightly.
    Args:
        ts: TrafficSignal object from sumo_rl.
    Returns:
        float: Total reward for the traffic signal.
    """
    reward = 0
    traci = ts.sumo
    for lane in ts.lanes:
        vehicles = traci.lane.getLastStepVehicleIDs(lane)
        for veh in vehicles:
            speed = traci.vehicle.getSpeed(veh)
            if speed < 0.1:
                v_type = traci.vehicle.getTypeID(veh)
                if v_type == "ambulance":
                    reward -= 50.0
                elif v_type == "bus":
                    reward -= 5.0
                else:
                    reward -= 1.0
    return reward

In [4]:
class PriorityObservationFunction(ObservationFunction):
    """
    Custom observation function for traffic signals.
    Returns lane densities and binary indicators for ambulance presence.
    """
    def __init__(self, ts):
        super().__init__(ts)

    def __call__(self):
        traci = self.ts.sumo
        obs = []
        density = self.ts.get_lanes_density()
        obs.extend(density)
        for lane in self.ts.lanes:
            emergency_waiting = 0.0
            vehicles = traci.lane.getLastStepVehicleIDs(lane)
            for veh in vehicles:
                if traci.vehicle.getTypeID(veh) == "ambulance":
                    emergency_waiting = 1.0
                    break
            obs.append(emergency_waiting)
        return np.array(obs, dtype=np.float32)

    def observation_space(self):
        return spaces.Box(low=0., high=1., shape=(len(self.ts.lanes) * 2,), dtype=np.float32)

In [5]:
# Initialize PettingZoo environment

base = Path("env_iiser") / "new"

route_files = [
    base / "vtypes.add.xml",
    base / "cars.rou.xml",
    base / "buses.rou.xml",
    base / "bikes.rou.xml",
    base / "ambulance.rou.xml",
]

net_path = str(base / "map.net.xml")

route = ",".join(str(p) for p in route_files)

env = sumo_rl.parallel_env(
    net_file=net_path,
    route_file=route,
    reward_fn=priority_reward_fn,
    observation_class=PriorityObservationFunction,
    use_gui=False,
    num_seconds=num_seconds,
    delta_time=5,
    sumo_warnings=False
)

In [6]:
# Get dimensions for the network
sample_agent = env.possible_agents[0]
obs_dim = env.observation_space(sample_agent).shape[0]
action_dim = env.action_space(sample_agent).n

print(f"Observation dimension: {obs_dim}")
print(f"Action dimension: {action_dim}")

Observation dimension: 6
Action dimension: 2


In [7]:
class QNetwork(nn.Module):
    """
    Q-Network for DQN.
    Two hidden layers with ReLU activation.
    """
    def __init__(self, obs_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [8]:
# Initialize networks, optimizers, and buffers
q_nets = {}
optimizers = {}
replay_buffers = {}
loss_fn = nn.MSELoss()

for agent in env.possible_agents:
    obs_dim = env.observation_space(agent).shape[0]
    action_dim = env.action_space(agent).n
    q_nets[agent] = QNetwork(obs_dim, action_dim)
    optimizers[agent] = optim.Adam(q_nets[agent].parameters(), lr=learning_rate)
    replay_buffers[agent] = deque(maxlen=replay_buffer_size)

In [9]:
# Initialize TensorBoard writer
writer = SummaryWriter(log_dir='runs/dqn_training')

all_episode_rewards = []

# Training loop with progress bar
for episode in tqdm(range(episodes), desc="Training Episodes"):
    observations, infos = env.reset()
    episode_reward = 0
    episode_losses = []
    
    while env.agents:
        actions = {}
        
        # Epsilon-Greedy Action Selection
        for agent in env.agents:
            obs_tensor = torch.FloatTensor(observations[agent]).unsqueeze(0)
            
            if random.random() < epsilon:
                actions[agent] = env.action_space(agent).sample()
            else:
                with torch.no_grad():
                    q_values = q_nets[agent](obs_tensor)
                    actions[agent] = torch.argmax(q_values).item()

        # Step the Environment 
        next_observations, rewards, terminations, truncations, infos = env.step(actions)

        # Store Transitions and Train
        for agent in env.agents:
            replay_buffers[agent].append((
                observations[agent], 
                actions[agent], 
                rewards[agent], 
                next_observations[agent], 
                terminations[agent] or truncations[agent]
            ))
            episode_reward += rewards[agent]

            if len(replay_buffers[agent]) > batch_size:
                batch = random.sample(replay_buffers[agent], batch_size)
                b_obs, b_act, b_rew, b_next_obs, b_done = zip(*batch)
                
                b_obs = torch.FloatTensor(np.array(b_obs))
                b_act = torch.LongTensor(b_act).unsqueeze(1)
                b_rew = torch.FloatTensor(b_rew).unsqueeze(1)
                b_next_obs = torch.FloatTensor(np.array(b_next_obs))
                b_done = torch.FloatTensor(b_done).unsqueeze(1)

                current_q = q_nets[agent](b_obs).gather(1, b_act)
                
                with torch.no_grad():
                    max_next_q = q_nets[agent](b_next_obs).max(1)[0].unsqueeze(1)
                    target_q = b_rew + (gamma * max_next_q * (1 - b_done))
                
                loss = loss_fn(current_q, target_q)
                optimizers[agent].zero_grad()
                loss.backward()
                optimizers[agent].step()
                
                episode_losses.append(loss.item())

        observations = next_observations

    all_episode_rewards.append(episode_reward)
    
    avg_loss = np.mean(episode_losses) if episode_losses else 0
    writer.add_scalar('Reward/Episode', episode_reward, episode)
    writer.add_scalar('Loss/Episode', avg_loss, episode)
    writer.add_scalar('Epsilon', epsilon, episode)
    
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    # Save models periodically
    if (episode + 1) % save_interval == 0:
        for agent in env.possible_agents:
            model_path = Path("models") / "iiser_priority" / f"{agent}_ep{episode+1}.pth"
            torch.save(q_nets[agent].state_dict(), str(model_path))

env.close()
writer.close()

Training Episodes:  24%|██▍       | 12/50 [09:34<30:18, 47.84s/it]


KeyboardInterrupt: 

In [ ]:
# Save final models
for agent in env.possible_agents:
    model_path = Path("models") / "iiser_priority" / f"{agent}.pth"
    torch.save(q_nets[agent].state_dict(), str(model_path))

In [ ]:
def plot_rewards(rewards, window_size=5):
    """
    Plot training rewards with moving average.
    """
    plt.figure(figsize=(10, 5))
    
    plt.plot(rewards, label='Raw Episode Reward', color='dodgerblue', alpha=0.5)
    
    if len(rewards) >= window_size:
        window = np.ones(window_size) / window_size
        
        smoothed_rewards = np.convolve(rewards, window, mode='valid')
        
        x_values = range(window_size - 1, len(rewards))
        
        plt.plot(x_values, smoothed_rewards, label=f'Moving Average (Window={window_size})', color='darkorange', linewidth=2)
    
    plt.title('Independent DQN Training')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.savefig('training_curve.png')
    plt.show()

plot_rewards(all_episode_rewards)

In [ ]:
def evaluate_model():
    """
    Evaluate the trained model in the environment with GUI.
    """
    env = sumo_rl.parallel_env(
        net_file=net_path,
        route_file=route,
        use_gui=True,
        num_seconds=600,
        delta_time=5,
        observation_class=PriorityObservationFunction,
        reward_fn=priority_reward_fn
    )

    eval_q_nets = {}
    for agent in env.possible_agents:
        obs_dim = env.observation_space(agent).shape[0]
        action_dim = env.action_space(agent).n
        eval_q_nets[agent] = QNetwork(obs_dim, action_dim)
        
        try:
            eval_q_nets[agent].load_state_dict(torch.load(str(Path("models") / "iiser_priority" / f"{agent}.pth")))
            eval_q_nets[agent].eval()
        except FileNotFoundError:
            print(f"Error: Could not find the model for {agent}.")
            return

    print("Models loaded successfully!")

    observations, infos = env.reset()
    total_evaluation_reward = 0
    
    while env.agents:
        actions = {}
        for agent in env.agents:
            obs_tensor = torch.FloatTensor(observations[agent]).unsqueeze(0)
            with torch.no_grad():
                q_values = eval_q_nets[agent](obs_tensor)
                actions[agent] = torch.argmax(q_values).item() 

        observations, rewards, terminations, truncations, infos = env.step(actions)

        for agent in env.agents:
            total_evaluation_reward += rewards[agent]

    print(f"Simulation Finished. Final Total Reward: {total_evaluation_reward:.2f}")
    env.close()

evaluate_model()